# NB10 — Final Evaluation & Comparison

**Purpose:** Produce the standardised evaluation table comparing ALL methods.

**Evaluation metrics:** Top-1 accuracy, ±1 step (clinical), MAE (Hz), per-patient breakdown,
metric–GT correlation.

**Runs on:** Local CPU

## 0. Install Dependencies

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'pydicom', 'numpy', 'pandas', 'matplotlib', 'Pillow', 'openpyxl', 'scipy'])
print('Dependencies OK')

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['figure.facecolor'] = '#0f0f1a'
matplotlib.rcParams['axes.facecolor']   = '#1a1a2e'
matplotlib.rcParams['text.color']       = 'white'
%matplotlib inline

GT_PATH = r'E:/CINE-Test1/1stcohortSAX (1).xlsx'
OUT_DIR = 'outputs'
EXCLUDE = ['Patient-KP']

from utils.gt_loader import load_gt, evaluate_predictions
from utils.vis_utils import plot_accuracy_bar, summary_text

gt_dict = load_gt(GT_PATH)
print('GT loaded.')

## 1. Load All Metric & Composite CSVs

In [ ]:
df_signal   = pd.read_csv(f'{OUT_DIR}/signal_metrics.csv')
df_contrast = pd.read_csv(f'{OUT_DIR}/contrast_metrics.csv')
df_banding  = pd.read_csv(f'{OUT_DIR}/banding_metrics.csv')
df_edge     = pd.read_csv(f'{OUT_DIR}/edge_metrics.csv')
df_temporal = pd.read_csv(f'{OUT_DIR}/temporal_metrics.csv')

df_comp_equal   = pd.read_csv(f'{OUT_DIR}/composite_equal.csv')
df_comp_physics = pd.read_csv(f'{OUT_DIR}/composite_physics.csv')
df_comp_grid    = pd.read_csv(f'{OUT_DIR}/composite_gridsearch.csv')

cnn_path = f'{OUT_DIR}/cnn_scores.csv'
has_cnn  = os.path.exists(cnn_path)
if has_cnn:
    df_cnn = pd.read_csv(cnn_path)
    print('CNN scores loaded.')
else:
    print('WARNING: cnn_scores.csv not found — CNN excluded from table.')

print('All CSVs loaded.')

## 2. Evaluate All Methods

In [ ]:
def eval_selector(df, score_col, argfn, name):
    pred_dict = {}
    for pid, grp in df.groupby('patient_id'):
        if pid in EXCLUDE: continue
        vals = grp.sort_values('image_no')[score_col].values
        pred_dict[pid] = int(argfn(vals)) + 1
    res = evaluate_predictions(pred_dict, gt_dict, exclude=EXCLUDE)
    n   = res['n_valid']
    return {
        'Method':    name,
        'Top-1':     f"{res['top1']*100:.1f}%  ({round(res['top1']*n)}/{n})",
        '±1 step':   f"{res['pm1']*100:.1f}%  ({round(res['pm1']*n)}/{n})",
        'MAE (Hz)':  f"{res['mae_hz']:.1f}",
        '_top1': res['top1'], '_pm1': res['pm1'], '_mae': res['mae_hz'], '_df': res['results_df'],
    }

results = []

# Individual metrics
results.append(eval_selector(df_signal,   'mu_myo',                 np.argmax, 'argmax(mu_myo)'))
results.append(eval_selector(df_signal,   'snr_myo',                np.argmax, 'argmax(snr_myo)'))
results.append(eval_selector(df_signal,   'homogeneity',            np.argmax, 'argmax(homogeneity)'))
results.append(eval_selector(df_contrast, 'cnr',                    np.argmax, 'argmax(cnr)'))
results.append(eval_selector(df_contrast, 'contrast_ratio',         np.argmax, 'argmax(contrast_ratio)'))
results.append(eval_selector(df_banding,  'banded_fraction_myo',    np.argmin, 'argmin(banded_fraction_myo)'))
results.append(eval_selector(df_banding,  'banding_burden',         np.argmin, 'argmin(banding_burden)'))
results.append(eval_selector(df_edge,     'edge_sharpness_boundary',np.argmax, 'argmax(edge_sharpness_boundary)'))
results.append(eval_selector(df_edge,     'laplacian_variance',     np.argmax, 'argmax(laplacian_variance)'))
results.append(eval_selector(df_temporal, 'temporal_std_sweep',     np.argmin, 'argmin(temporal_std_sweep)'))
results.append(eval_selector(df_temporal, 'local_signal_drop',      np.argmin, 'argmin(local_signal_drop)'))
if has_cnn:
    results.append(eval_selector(df_cnn, 'cnn_score', np.argmax, 'argmax(cnn_score) EfficientNet-B0'))

# Composite
results.append(eval_selector(df_comp_equal,   'composite_score', np.argmax, 'Composite — equal weight'))
results.append(eval_selector(df_comp_physics, 'composite_score', np.argmax, 'Composite — physics-informed'))
results.append(eval_selector(df_comp_grid,    'composite_score', np.argmax, 'Composite — grid search (in-sample)'))

# Reference
results.append({'Method': '=== Prior pipeline (Exp10) ===',
                'Top-1': '42.3%  (11/26)', '±1 step': '76.9%  (20/26)', 'MAE (Hz)': '22.1',
                '_top1': 0.423, '_pm1': 0.769, '_mae': 22.1, '_df': None})

for r in results:
    print(f"  {r['Method'][:50]:50s} Top-1:{r['Top-1']:22s} ±1:{r['±1 step']:17s} MAE:{r['MAE (Hz)']}")

## 3. Comparison Table & Bar Chart

In [ ]:
TABLE_COLS = ['Method', 'Top-1', '±1 step', 'MAE (Hz)']
table_df = pd.DataFrame([{c: r[c] for c in TABLE_COLS} for r in results])
print('\n=== FULL COMPARISON TABLE ===')
print(table_df.to_string(index=False))
table_df.to_csv(f'{OUT_DIR}/final_comparison_table.csv', index=False)

# Bar chart
fig = plot_accuracy_bar(
    methods=[r['Method'] for r in results],
    top1_vals=[r['_top1'] for r in results],
    pm1_vals=[r['_pm1'] for r in results],
    title='Method Comparison — bSSFP Frequency Scout Selection',
    figsize=(12, max(6, len(results)*0.5)),
    save_path=f'{OUT_DIR}/final_comparison_bar.png',
)
plt.show()
print('Saved final_comparison_bar.png')

## 4. Per-Patient Breakdown

In [ ]:
breakdown_data = {}
for comp_name, comp_df in [('equal', df_comp_equal), ('physics', df_comp_physics), ('grid', df_comp_grid)]:
    pred_dict = {}
    for pid, grp in comp_df.groupby('patient_id'):
        if pid in EXCLUDE: continue
        pred_dict[pid] = int(np.argmax(grp.sort_values('image_no')['composite_score'].values)) + 1
    res = evaluate_predictions(pred_dict, gt_dict, exclude=EXCLUDE)
    breakdown_data[comp_name] = res['results_df'].set_index('patient_id')

rows = []
for pid in sorted(breakdown_data['equal'].index):
    gt_no = gt_dict[pid]['image_no']
    row   = {'Patient': pid, 'GT_no': gt_no}
    for comp_name, df_res in breakdown_data.items():
        if pid in df_res.index:
            r      = df_res.loc[pid]
            status = '✅' if r['correct'] else ('~±1' if r['pm1'] else '❌')
            row[f'Pred_{comp_name}'] = f"#{int(r['pred_no'])} {status}"
    rows.append(row)

breakdown_df = pd.DataFrame(rows)
print(breakdown_df.to_string(index=False))
breakdown_df.to_csv(f'{OUT_DIR}/per_patient_breakdown.csv', index=False)

## 5. Metric–GT Correlation Analysis

In [ ]:
KEY = ['patient_id', 'image_no']
master = df_signal[KEY + ['mu_myo', 'snr_myo', 'homogeneity']].copy()
master = master.merge(df_contrast[KEY + ['cnr']], on=KEY)
master = master.merge(df_banding[KEY  + ['banded_fraction_myo', 'banding_burden']], on=KEY)
master = master.merge(df_edge[KEY     + ['edge_sharpness_boundary']], on=KEY)
master = master.merge(df_temporal[KEY + ['temporal_std_sweep']], on=KEY)
master = master[~master['patient_id'].isin(EXCLUDE)]

def is_gt(row):
    return int(gt_dict.get(row['patient_id'], {}).get('image_no', -1) == row['image_no'])
master['is_gt'] = master.apply(is_gt, axis=1)

corr_cols = ['mu_myo', 'snr_myo', 'homogeneity', 'cnr',
             'banded_fraction_myo', 'banding_burden', 'edge_sharpness_boundary', 'temporal_std_sweep']

corr_results = []
for col in corr_cols:
    c = master[col].corr(master['is_gt'])
    corr_results.append({'Metric': col, 'Corr_with_GT': round(c, 4),
                          'Direction': '↑ higher=better' if c > 0 else '↓ lower=better'})

corr_df = pd.DataFrame(corr_results).sort_values('Corr_with_GT', key=abs, ascending=False)
print('Pearson correlation with GT label (1=GT, 0=not GT):')
print(corr_df.to_string(index=False))
corr_df.to_csv(f'{OUT_DIR}/metric_gt_correlations.csv', index=False)

## 6. Results Table for README

In [ ]:
print('\n=== Copy this into README.md ===')
print()
print('| Method | Top-1 | ±1 step | MAE (Hz) |')
print('|---|---|---|---|')
for r in results:
    print(f"| {r['Method']} | {r['Top-1']} | {r['±1 step']} | {r['MAE (Hz)']} |")

print('\n✅ NB10 complete!')
print('Next: Plug the best composite weights into freqscout_agent for Exp11/Exp12')